In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.datasets import fashion_mnist

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

In [ ]:
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [ ]:
cl = {
   0  :"T-shirt/top", 
   1  :"Trouser",
   2  :"Pullover", 
   3  :"Dress",
   4  :"Coat", 
   5  :"Sandal", 
   6  :"Shirt", 
   7  :"Sneaker", 
   8  :"Bag",
   9  :"Ankle boot" }

In [ ]:
plt.figure(figsize=(10, 15))
for i in range(20):
    plt.subplot(5, 4, i+1)
    index = np.random.randint(0, x_train.shape[0]-1)
    plt.imshow(x_train[index], cmap="gray")
    plt.xticks([])
    plt.yticks([])
    plt.title(cl[y_train[index]])

In [ ]:
x_train = x_train / 255.
x_test = x_test / 255.
y_train_k = tf.keras.utils.to_categorical(y_train)
y_test_k = tf.keras.utils.to_categorical(y_test)

In [ ]:
model = tf.keras.Sequential([
    Input(x_train.shape[1:], 32),
    Flatten(),
    Dense(1000, activation='sigmoid'),
    Dense(1000, activation='relu'),
    Dense(10, activation='softmax')
])
model.summary()

In [ ]:
model.compile(optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']) #0.8748, 0.8819 

In [ ]:
model.fit(x_train, y_train_k, 32, , validation_data=(x_test, y_test_k))

In [ ]:
preds = model.predict(x_test).argmax(axis=1)

In [ ]:
mask = (preds != y_test)

In [ ]:
x_fails = x_test[mask]
y_fails = y_test[mask]
preds_fails = preds[mask]

In [ ]:
plt.figure(figsize=(10, 15))
for i in range(20):
    plt.subplot(5, 4, i+1)
    index = np.random.randint(0, x_fails.shape[0]-1)
    plt.imshow(x_fails[index], cmap="gray")
    plt.xticks([])
    plt.yticks([])
    plt.title(f"real : {cl[y_fails[index]]}\npred : {cl[preds_fails[index]]}")

In [ ]:
def predict_num(img_path, k=5):
    img = Image.open(img_path)
    img = img.convert("L").resize((28, 28)).crop((0,8, 27, 27)).resize((28, 28))
    img = (255. - np.asarray(img)) / 255.
    plt.imshow(img, cmap="gray")
    p_img = np.expand_dims(img, axis=0)
    res = model.predict(p_img)[0]
    cls = np.argsort(res)[::-1]
    pr = np.sort(res)[::-1]
    for i in range(k):
        print(f"{cl[cls[i]]} - {pr[i]:.5f}")
    return np.argmax(res)

In [ ]:
predict_num('sneaker.jpg')